In [3]:
from pydrake.all import FindResourceOrThrow, StartMeshcat, MeshcatVisualizer
import numpy as np
meshcat = StartMeshcat()

INFO:drake:Meshcat listening for connections at http://localhost:7003


In [7]:
import importlib, io, contextlib
import numpy as np
ftt = importlib.import_module("find_throwing_trajectory")
importlib.reload(ftt)

N = 5
initial_state = np.zeros(4)
final_configuration = np.array([np.pi, 0])
tf = 4
distance = 15
max_tries = 50
warm_file = f"warm_start_N{N}_d{distance}_tf{tf}.npz"

# Wrap Solve so we can grab the result (and optionally warm start) without editing the graded function
_orig_Solve = ftt.Solve
captured = {}
warm = None   # set to np.load(warm_file)["z"] to warm start every try from a saved solution
def _Solve(prog, *args, **kwargs):
  if warm is not None:
    prog.SetInitialGuess(prog.decision_variables(), warm)
  result = _orig_Solve(prog, *args, **kwargs)
  captured["prog"], captured["result"] = prog, result
  return result
ftt.Solve = _Solve

x_traj = u_traj = prog = None
for attempt in range(1, max_tries + 1):
  with contextlib.redirect_stdout(io.StringIO()):   # silence per-run prints
    x_traj, u_traj, prog, _, _ = ftt.find_throwing_trajectory(N, initial_state, final_configuration, distance, tf)
  result = captured["result"]
  print(f"try {attempt}: {result.get_solution_result()}")
  if result.is_success():
    z = result.GetSolution(captured["prog"].decision_variables())   # [x0,u0,x1,u1,...,t_land]
    np.savez(warm_file, z=z, N=N, distance=distance, tf=tf)
    print(f"solved on try {attempt}, cost {result.get_optimal_cost():.2f}, saved {warm_file}")
    break
else:
  print(f"no solution in {max_tries} tries")
ftt.Solve = _orig_Solve


try 1: SolutionResult.kInfeasibleConstraints
try 2: SolutionResult.kInfeasibleConstraints
try 3: SolutionResult.kInfeasibleConstraints
try 4: SolutionResult.kInfeasibleConstraints
try 5: SolutionResult.kInfeasibleConstraints
try 6: SolutionResult.kInfeasibleConstraints
try 7: SolutionResult.kInfeasibleConstraints
try 8: SolutionResult.kInfeasibleConstraints
try 9: SolutionResult.kInfeasibleConstraints
try 10: SolutionResult.kInfeasibleConstraints
try 11: SolutionResult.kIterationLimit
try 12: SolutionResult.kInfeasibleConstraints
try 13: SolutionResult.kInfeasibleConstraints
try 14: SolutionResult.kInfeasibleConstraints
try 15: SolutionResult.kInfeasibleConstraints
try 16: SolutionResult.kInfeasibleConstraints
try 17: SolutionResult.kInfeasibleConstraints
try 18: SolutionResult.kInfeasibleConstraints
try 19: SolutionResult.kInfeasibleConstraints
try 20: SolutionResult.kInfeasibleConstraints
try 21: SolutionResult.kInfeasibleConstraints
try 22: SolutionResult.kSolutionFound
solved on tr

In [8]:
import matplotlib.pyplot as plt
import numpy as np
import time

from pydrake.all import (
    DiagramBuilder, Simulator, FindResourceOrThrow, MultibodyPlant, PiecewisePolynomial, SceneGraph,
    Parser, MultibodyPositionToGeometryPose, TrajectorySource, Demultiplexer, ConstantVectorSource,
    Sphere, Box, Rgba, RigidTransform
)


# Create a MultibodyPlant for the arm
file_name = "planar_arm.urdf"
builder = DiagramBuilder()
scene_graph = builder.AddSystem(SceneGraph())
planar_arm = builder.AddSystem(MultibodyPlant(0.0))
planar_arm.RegisterAsSourceForSceneGraph(scene_graph)
Parser(plant=planar_arm).AddModels(file_name)
planar_arm.Finalize()

n_q = planar_arm.num_positions()
n_v = planar_arm.num_velocities()
n_u = planar_arm.num_actuators()


x_traj_source = builder.AddSystem(TrajectorySource(x_traj))
u_traj_source = builder.AddSystem(TrajectorySource(u_traj))

demux = builder.AddSystem(Demultiplexer(np.array([n_q, n_v])))
to_pose = builder.AddSystem(MultibodyPositionToGeometryPose(planar_arm))
zero_inputs = builder.AddSystem(ConstantVectorSource(np.zeros(n_u)))

builder.Connect(zero_inputs.get_output_port(), planar_arm.get_actuation_input_port())
builder.Connect(x_traj_source.get_output_port(), demux.get_input_port())
builder.Connect(demux.get_output_port(0), to_pose.get_input_port())
builder.Connect(to_pose.get_output_port(), scene_graph.get_source_pose_port(planar_arm.get_source_id()))

MeshcatVisualizer.AddToBuilder(builder, scene_graph, meshcat)

diagram = builder.Build()
diagram.set_name("diagram")


# Release state (planar x,z from arm kinematics) -> ballistic flight
g, l = 9.81, 1.0
xf = x_traj.value(tf).ravel()
q, qd = xf[:2], xf[2:]
p0 = np.array([-l*np.sin(q[0]) - l*np.sin(q[0]+q[1]),
               -l*np.cos(q[0]) - l*np.cos(q[0]+q[1])])
v0 = np.array([-l*qd[1]*np.cos(q[0]+q[1]) + qd[0]*(-l*np.cos(q[0]) - l*np.cos(q[0]+q[1])),
                l*qd[1]*np.sin(q[0]+q[1]) + qd[0]*( l*np.sin(q[0]) + l*np.sin(q[0]+q[1]))])
t_flight = (v0[1] + np.sqrt(v0[1]**2 + 2*g*p0[1])) / g   # positive root of z(t) = 0
y_plane = 0.25                                            # out-of-plane offset from URDF
print("lands at x =", p0[0] + v0[0]*t_flight, " target:", distance)

meshcat.SetObject("thrown_ball", Sphere(0.1), Rgba(1, 0.5, 0, 1))
meshcat.SetObject("target", Box(0.1, 0.4, 0.02), Rgba(0, 0.8, 0, 1))
meshcat.SetTransform("target", RigidTransform([distance, y_plane, 0]))

# Record arm playback + ball flight; replay from the Animations panel in Meshcat
meshcat.StartRecording()
simulator = Simulator(diagram)
simulator.Initialize()
simulator.set_target_realtime_rate(1)
simulator.AdvanceTo(tf)

fps = 60
meshcat.SetProperty("thrown_ball", "visible", False, 0.0)
meshcat.SetProperty("thrown_ball", "visible", True, tf)
for t in np.arange(0, t_flight, 1/fps):
  pos = [p0[0] + v0[0]*t, y_plane, p0[1] + v0[1]*t - 0.5*g*t**2]
  meshcat.SetTransform("thrown_ball", RigidTransform(pos), tf + t)
meshcat.StopRecording()
meshcat.PublishRecording()


lands at x = 15.000000000102307  target: 15
